# Multilingual RAG retrieval (English + Hindi)

Cross-lingual retrieval over the **Universal Declaration of Human Rights (UDHR)** using **multilingual-e5** embeddings and **LangChain + Chroma**.

**Prerequisites** (from repo root):

```bash
uv sync
uv run jupyter lab experiments/multilingual-rag-retrieval/multilingual_e5_chroma.ipynb
```

Run cells top to bottom.

## PDF sources

Place PDFs in `data/` (next to this notebook):

| File | Language | Official download |
|------|----------|-------------------|
| `udhr_en.pdf` | English | [OHCHR eng.pdf](https://www.ohchr.org/sites/default/files/UDHR/Documents/UDHR_Translations/eng.pdf) |
| `udhr_hi.pdf` | Hindi | [OHCHR hnd.pdf](https://www.ohchr.org/sites/default/files/UDHR/Documents/UDHR_Translations/hnd.pdf) |

Chroma index is stored under `workspace/chroma/multilingual_rag_retrieval/` (gitignored).

In [5]:
# Optional: download missing PDFs from OHCHR (skip if you already have local files)
from pathlib import Path
from urllib.request import Request, urlopen

DATA_RAW = Path("data")
DATA_RAW.mkdir(parents=True, exist_ok=True)

OHCHR_PDFS = {
    "udhr_en.pdf": "https://www.ohchr.org/sites/default/files/UDHR/Documents/UDHR_Translations/eng.pdf",
    "udhr_hi.pdf": "https://www.ohchr.org/sites/default/files/UDHR/Documents/UDHR_Translations/hnd.pdf",
}

DOWNLOAD_MISSING = False  # set True to fetch any file not in data/raw/

def _download(url: str, dest: Path) -> None:
    req = Request(url, headers={"User-Agent": "awesome-rag-experiments/1.0"})
    with urlopen(req) as resp, dest.open("wb") as out:
        out.write(resp.read())


if DOWNLOAD_MISSING:
    for name, url in OHCHR_PDFS.items():
        path = DATA_RAW / name
        if not path.exists():
            print(f"Downloading {name}...")
            _download(url, path)
        else:
            print(f"Already present: {name}")
else:
    present = [p.name for p in DATA_RAW.glob("*.pdf")]
    print("PDFs in data/raw/:", present or "(none — add udhr_en.pdf and udhr_hi.pdf)")

PDFs in data/raw/: (none — add udhr_en.pdf and udhr_hi.pdf)


## 1. Setup + ingest

In [2]:
import re
import shutil
from pathlib import Path
from typing import Any

import pandas as pd
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "data" / "raw").exists():
    EXPERIMENT_DIR = Path("experiments/multilingual-rag-retrieval").resolve()

DATA_RAW = EXPERIMENT_DIR / "data" / "raw"
PROJECT_ROOT = EXPERIMENT_DIR.parent.parent
CHROMA_PATH = PROJECT_ROOT / "workspace" / "chroma" / "multilingual_rag_retrieval"
COLLECTION_NAME = "multilingual_rag_facts"
MODEL_NAME = "intfloat/multilingual-e5-base"

LANG_PDF_MAP = {
    "en": "udhr_en.pdf",
    "hi": "udhr_hi.pdf",
}

ARTICLE_MARKERS = {
    "en": re.compile(r"(?=\bArticle\s+(\d+)\b)", re.IGNORECASE),
    "hi": re.compile(r"(?=अनुच्छेद\s*(\d+))"),
}
ARTICLE_ID_FROM_START = {
    "en": re.compile(r"^\s*Article\s+(\d+)\b", re.IGNORECASE),
    "hi": re.compile(r"^\s*अनुच्छेद\s*(\d+)"),
}


class E5Embeddings(Embeddings):
    """multilingual-e5 requires query:/passage: prefixes."""

    def __init__(self, model_name: str = MODEL_NAME) -> None:
        self._model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        prefixed = [f"passage: {t}" for t in texts]
        vectors = self._model.encode(
            prefixed, normalize_embeddings=True, show_progress_bar=True
        )
        return vectors.tolist()

    def embed_query(self, text: str) -> list[float]:
        vectors = self._model.encode(
            [f"query: {text}"], normalize_embeddings=True, show_progress_bar=False
        )
        return vectors[0].tolist()


def _article_id(lang: str, chunk: str) -> str:
    pattern = ARTICLE_ID_FROM_START.get(lang)
    if pattern:
        match = pattern.search(chunk.strip())
        if match:
            return match.group(1)
    return "preamble"


def load_lang_pdf(lang: str) -> str:
    filename = LANG_PDF_MAP[lang]
    path = DATA_RAW / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing {path} — add PDF or set DOWNLOAD_MISSING=True above")
    pages = PyPDFLoader(str(path)).load()
    return "\n".join(page.page_content for page in pages)


def split_by_articles(lang: str, text: str) -> list[Document]:
    marker = ARTICLE_MARKERS.get(lang)
    if marker:
        parts = marker.split(text)
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) > 1:
            docs = []
            for part in parts:
                article = _article_id(lang, part)
                docs.append(
                    Document(
                        page_content=part,
                        metadata={"lang": lang, "article": article, "source": "udhr"},
                    )
                )
            return docs

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    return splitter.create_documents(
        [text],
        metadatas=[{"lang": lang, "article": "unknown", "source": "udhr"}],
    )


def build_documents(langs: list[str]) -> list[Document]:
    all_docs: list[Document] = []
    for lang in langs:
        text = load_lang_pdf(lang)
        chunks = split_by_articles(lang, text)
        print(f"{lang}: {len(chunks)} chunks")
        all_docs.extend(chunks)
    return all_docs


def build_vectorstore(documents: list[Document], *, reset: bool = True) -> Chroma:
    if reset and CHROMA_PATH.exists():
        shutil.rmtree(CHROMA_PATH)
    CHROMA_PATH.mkdir(parents=True, exist_ok=True)
    embeddings = E5Embeddings()
    return Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(CHROMA_PATH),
    )


print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Chroma path: {CHROMA_PATH}")

/var/folders/gk/4bnsrx_909dfbhh45n_x91c00000gn/T/ipykernel_9799/2174257739.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Experiment dir: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/multilingual-rag-retrieval
Chroma path: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/workspace/chroma/multilingual_rag_retrieval


In [3]:
# Languages to ingest: English + Hindi
langs = ["en", "hi"]

documents = build_documents(langs)
vectorstore = build_vectorstore(documents, reset=True)
print(f"Indexed {len(documents)} chunks into '{COLLECTION_NAME}'")

FileNotFoundError: Missing /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/multilingual-rag-retrieval/data/raw/udhr_en.pdf — add PDF or set DOWNLOAD_MISSING=True above

## 2. Search

Change `query` and `query_lang`, then run the cell. A hit in another language with the same article is cross-lingual retrieval.

In [ ]:
query = "Everyone has the right to life, liberty and security of person."
query_lang = "en"  # en | hi
k = 5

results = vectorstore.similarity_search_with_score(query, k=k)

rows = []
for rank, (doc, score) in enumerate(results, start=1):
    rows.append(
        {
            "rank": rank,
            "article": doc.metadata.get("article"),
            "lang": doc.metadata.get("lang"),
            "score": round(float(score), 4),
            "text": doc.page_content[:120] + "...",
        }
    )

display(pd.DataFrame(rows))

## 3. Evaluate

**Metric:** Recall@k and MRR. A query **hits** if top-k contains the same `article` in a **different** language than `query_lang`.

In [ ]:
EVAL_QUERIES = [
    {
        "query": "All human beings are born free and equal in dignity and rights.",
        "query_lang": "en",
        "expected_article": "1",
    },
    {
        "query": "Everyone has the right to life, liberty and security of person.",
        "query_lang": "en",
        "expected_article": "3",
    },
    {
        "query": "Everyone has the right to freedom of opinion and expression.",
        "query_lang": "en",
        "expected_article": "19",
    },
    {
        "query": "सभी मनुष्य जन्म से स्वतंत्र और समान गरिमा और अधिकारों में हैं।",
        "query_lang": "hi",
        "expected_article": "1",
    },
    {
        "query": "हर किसी को जीवन, स्वतंत्रता और व्यक्तिगत सुरक्षा का अधिकार है।",
        "query_lang": "hi",
        "expected_article": "3",
    },
    {
        "query": "हर किसी को राय और अभिव्यक्ति की स्वतंत्रता का अधिकार है।",
        "query_lang": "hi",
        "expected_article": "19",
    },
]


def is_cross_lingual_hit(doc: Document, expected_article: str, query_lang: str) -> bool:
    return (
        doc.metadata.get("article") == expected_article
        and doc.metadata.get("lang") != query_lang
    )


def evaluate(vectorstore: Chroma, queries: list[dict[str, Any]], k: int = 5) -> dict[str, Any]:
    per_query = []
    reciprocal_ranks: list[float] = []
    hits = 0

    for item in queries:
        results = vectorstore.similarity_search_with_score(item["query"], k=k)
        rr = 0.0
        hit = False
        for rank, (doc, _) in enumerate(results, start=1):
            if is_cross_lingual_hit(doc, item["expected_article"], item["query_lang"]):
                hit = True
                rr = 1.0 / rank
                break
        if hit:
            hits += 1
        reciprocal_ranks.append(rr)
        per_query.append(
            {
                "query_lang": item["query_lang"],
                "expected_article": item["expected_article"],
                "hit": hit,
                "reciprocal_rank": rr,
                "query": item["query"][:60] + "...",
            }
        )

    n = len(queries) or 1
    return {
        "k": k,
        "recall_at_k": hits / n,
        "mrr": sum(reciprocal_ranks) / n,
        "per_query": per_query,
    }


metrics = evaluate(vectorstore, EVAL_QUERIES, k=5)
print(f"Recall@{metrics['k']}: {metrics['recall_at_k']:.1%}")
print(f"MRR: {metrics['mrr']:.3f}")
display(pd.DataFrame(metrics["per_query"]))